This version contains roughly 3 years of hourly data for about 30 stocks.

In [2]:
import requests
import pandas as pd
from pathlib import Path
import time

def load_api_key(filepath="api_keys/twelvedata.txt"):
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_twelve_data_hourly(ticker, api_key, outputsize=5000):
    url = "https://api.twelvedata.com/time_series"

    params = {
        "symbol": ticker,
        "interval": "1h",
        "outputsize": outputsize,
        "apikey": api_key,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if "values" not in data:
        raise ValueError(f"Twelve Data error for {ticker}: {data}")

    df = pd.DataFrame(data["values"])

    df = df.rename(columns={
        "datetime": "timestamp",
        "open": "open",
        "high": "high",
        "low": "low",
        "close": "close",
        "volume": "volume",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ticker"] = ticker

    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["vwap"] = pd.NA
    df["transactions"] = pd.NA

    df = df[
        ["timestamp", "ticker", "open", "high", "low", "close", "volume", "vwap", "transactions"]
    ]

    return df.sort_values("timestamp").reset_index(drop=True)


def download_twelve_data_hourly_for_tickers(
    tickers,
    api_key_path="api_keys/twelve_data.txt",
    save_path="raw_hourly_data.csv",
    sleep_seconds=8
):
    api_key = load_api_key(api_key_path)
    all_dfs = []

    for ticker in tickers:
        print(f"Downloading {ticker}...")
        df = get_twelve_data_hourly(ticker, api_key)
        all_dfs.append(df)
        time.sleep(sleep_seconds)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.sort_values(["ticker", "timestamp"]).reset_index(drop=True)

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    combined_df.to_csv(save_path, index=False)

    print(f"Saved {len(combined_df):,} rows to {save_path}")

    return combined_df

In [8]:
tickers = [
    # Big Tech / Growth
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "TSLA", "NFLX", "META",
    
    # Finance
    "JPM", "BAC", "GS", "MS",
    
    # Consumer / Retail
    "WMT", "COST", "HD", "NKE", "SBUX",
    
    # Healthcare / Pharma
    "JNJ", "PFE", "MRK", "UNH",
    
    # Energy
    "XOM", "CVX",
    
    # Industrials / Transportation
    "BA", "CAT", "GE", "UPS",
    
    # ETFs (nice for comparison)
    "SPY", "QQQ", "DIA"
]
hourly_df = download_twelve_data_hourly_for_tickers(
    tickers=tickers,
    api_key_path="api_keys/twelvedata.txt",
    save_path="raw_hourly_data.csv",
    sleep_seconds=8
)

hourly_df.head()

Saved 150,000 rows to data/raw_hourly_data.csv


,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2023-06-20 09:30:00,AAPL,184.789990,186.100010,184.70000,184.810000,12001675,NaN,NaN
1,2023-06-20 10:30:00,AAPL,184.820010,185.360000,184.50999,185.060104,5824330,NaN,NaN
2,2023-06-20 11:30:00,AAPL,185.070007,185.600010,184.89999,185.554990,4348851,NaN,NaN
3,2023-06-20 12:30:00,AAPL,185.550000,186.089996,185.49001,185.929990,4413993,NaN,NaN
4,2023-06-20 13:30:00,AAPL,185.920000,186.029999,185.41499,185.770000,3535526,NaN,NaN


In [10]:
hourly_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2023-06-20 09:30:00,AAPL,184.789990,186.100010,184.70000,184.810000,12001675,NaN,NaN
1,2023-06-20 10:30:00,AAPL,184.820010,185.360000,184.50999,185.060104,5824330,NaN,NaN
2,2023-06-20 11:30:00,AAPL,185.070007,185.600010,184.89999,185.554990,4348851,NaN,NaN
3,2023-06-20 12:30:00,AAPL,185.550000,186.089996,185.49001,185.929990,4413993,NaN,NaN
4,2023-06-20 13:30:00,AAPL,185.920000,186.029999,185.41499,185.770000,3535526,NaN,NaN
...,...,...,...,...,...,...,...,...,...
149995,2026-04-28 11:30:00,XOM,151.890000,152.310000,151.57001,151.730000,1474558,NaN,NaN
149996,2026-04-28 12:30:00,XOM,151.730000,151.845000,151.39000,151.420000,1118399,NaN,NaN
149997,2026-04-28 13:30:00,XOM,151.420000,151.460010,150.75000,150.945010,1052452,NaN,NaN
149998,2026-04-28 14:30:00,XOM,150.945010,150.990010,150.34000,150.800000,1028051,NaN,NaN


In [9]:
hourly_df['ticker'].value_counts()

ticker
AAPL     5000
AMZN     5000
WMT      5000
UPS      5000
UNH      5000
TSLA     5000
SPY      5000
SBUX     5000
QQQ      5000
PFE      5000
NVDA     5000
NKE      5000
NFLX     5000
MSFT     5000
MS       5000
MRK      5000
META     5000
JPM      5000
JNJ      5000
HD       5000
GS       5000
GOOGL    5000
GE       5000
DIA      5000
CVX      5000
COST     5000
CAT      5000
BAC      5000
BA       5000
XOM      5000
Name: count, dtype: int64

In [11]:
# Temp for testing
hourly_df.to_csv('All Hourly Data.csv', index=False)

In [2]:
import pandas as pd
df = pd.read_csv('All Hourly Data.csv')
df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2023-06-20 09:30:00,AAPL,184.789990,186.100010,184.70000,184.810000,12001675,NaN,NaN
1,2023-06-20 10:30:00,AAPL,184.820010,185.360000,184.50999,185.060104,5824330,NaN,NaN
2,2023-06-20 11:30:00,AAPL,185.070007,185.600010,184.89999,185.554990,4348851,NaN,NaN
3,2023-06-20 12:30:00,AAPL,185.550000,186.089996,185.49001,185.929990,4413993,NaN,NaN
4,2023-06-20 13:30:00,AAPL,185.920000,186.029999,185.41499,185.770000,3535526,NaN,NaN
...,...,...,...,...,...,...,...,...,...
149995,2026-04-28 11:30:00,XOM,151.890000,152.310000,151.57001,151.730000,1474558,NaN,NaN
149996,2026-04-28 12:30:00,XOM,151.730000,151.845000,151.39000,151.420000,1118399,NaN,NaN
149997,2026-04-28 13:30:00,XOM,151.420000,151.460010,150.75000,150.945010,1052452,NaN,NaN
149998,2026-04-28 14:30:00,XOM,150.945010,150.990010,150.34000,150.800000,1028051,NaN,NaN


In [3]:
df['ticker'].value_counts()

ticker
AAPL     5000
AMZN     5000
WMT      5000
UPS      5000
UNH      5000
TSLA     5000
SPY      5000
SBUX     5000
QQQ      5000
PFE      5000
NVDA     5000
NKE      5000
NFLX     5000
MSFT     5000
MS       5000
MRK      5000
META     5000
JPM      5000
JNJ      5000
HD       5000
GS       5000
GOOGL    5000
GE       5000
DIA      5000
CVX      5000
COST     5000
CAT      5000
BAC      5000
BA       5000
XOM      5000
Name: count, dtype: int64

In [6]:
df.head(25)

,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2023-06-20 09:30:00,AAPL,184.789990,186.100010,184.700000,184.810000,12001675,NaN,NaN
1,2023-06-20 10:30:00,AAPL,184.820010,185.360000,184.509990,185.060104,5824330,NaN,NaN
2,2023-06-20 11:30:00,AAPL,185.070007,185.600010,184.899990,185.554990,4348851,NaN,NaN
3,2023-06-20 12:30:00,AAPL,185.550000,186.089996,185.490010,185.929990,4413993,NaN,NaN
4,2023-06-20 13:30:00,AAPL,185.920000,186.029999,185.414990,185.770000,3535526,NaN,NaN
5,2023-06-20 14:30:00,AAPL,185.770000,185.845000,185.130000,185.340000,3522844,NaN,NaN
6,2023-06-20 15:30:00,AAPL,185.340000,185.500000,184.810000,185.020004,4861772,NaN,NaN
7,2023-06-21 09:30:00,AAPL,184.899990,185.410000,183.870000,184.120000,10613238,NaN,NaN
8,2023-06-21 10:30:00,AAPL,184.110000,184.140000,182.590100,183.140000,7350639,NaN,NaN
9,2023-06-21 11:30:00,AAPL,183.130000,184.080002,183.085007,184.065002,4681627,NaN,NaN


In [7]:
df['ticker'].nunique()

30

# Try again to get dataframes for each